# Serie unificate delle presenze turistiche per comune

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import geopandas as geopd
import plotly.express as px

REPO_ROOT = Path.cwd() if (Path.cwd() / "utils.py").exists() else Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from utils import get_s3, get_mapping

## Download of the data and processing
#### Note: only the TOURISTS are taken into account in the following analysis

In [ ]:
vodafone_attendences = pd.read_csv(get_s3("vodafone_attendences.csv"))
vodafone_attendences_new = pd.read_csv(get_s3("vodafone_attendences_new.csv"))

geojson_comuni_json_data = geopd.read_file(get_s3("TRENTINO-comuni_Vodafone_2023.geojson"))
json_vodafone = get_mapping("mapping_comuni_into_vodafone_Trento.json")

In [ ]:
location_map = dict(
    zip(
        geojson_comuni_json_data["id"].astype(str),
        geojson_comuni_json_data["name"].str.upper(),
    )
)

vodafone_attendences['date'] = pd.to_datetime(vodafone_attendences['date'])
vodafone_attendences_new['date'] = pd.to_datetime(vodafone_attendences_new['date'])
vodafone_attendences_new = vodafone_attendences_new.sort_values('date').reset_index(drop=True)

vodafone_attendences_merged_comuni = vodafone_attendences_new[
    vodafone_attendences_new["locType"] == "TN_MKT_AL_3"
].copy()

vodafone_attendences_merged_comuni["comune"] = vodafone_attendences_merged_comuni["locId"].astype(str).map(location_map)
vodafone_attendences_merged_comuni["ID_COMUNE"] = vodafone_attendences_merged_comuni["comune"].map(json_vodafone)
mask = vodafone_attendences_merged_comuni["comune"].isin(["VIGO DI FASSA", "POZZA DI FASSA"])
vodafone_attendences_merged_comuni.loc[mask, "comune"] = "SAN GIOVANNI DI FASSA" # Vigo di Fassa e Pozza di Fassa -> San Giovanni di Fassa
vodafone_attendences_merged_comuni.loc[mask, "ID_COMUNE"] = pd.Series(
    [[22250]] * mask.sum(), index=vodafone_attendences_merged_comuni.index[mask]  
)

tourists_attendences = vodafone_attendences_merged_comuni[vodafone_attendences_merged_comuni['userProfile'] == 'TOURIST'].groupby(['date', 'comune']).agg({'value': 'sum', 'ID_COMUNE':'first'}).reset_index()

In [ ]:
# tourists_attendences = tourists_attendences[tourists_attendences.date >= pd.Timestamp('2023-01-01')]
tourists_attendences = tourists_attendences.pivot(index="date", columns="comune", values="value").asfreq("D").fillna(0).stack().reset_index(name="value")  # giorni mancanti = nessun turista -> 0
tourists_attendences["value"] = tourists_attendences.groupby("comune")["value"].transform(lambda s: s.rolling(7, center=True).mean())
tourists_attendences

## Serie unificate: tutti i comuni

In [ ]:
fig = px.line(
    tourists_attendences,
    x="date",
    y="value",
    color="comune",
    render_mode="webgl",
    labels={"date": "Data", "value": "Presenze turistiche", "comune": "Comune"},
    title="Presenze turistiche giornaliere per comune (serie unificate)",
    template="plotly_white",
)
fig.update_traces(line_width=1)
fig.update_layout(height=750)
fig.show()

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.signal import find_peaks, peak_widths

ts = tourists_attendences[tourists_attendences['comune']=='BORGO VALSUGANA']
ts = ts.set_index('date')['value']

def show_periodi(ts, title="BORGO VALSUGANA"):
    WINDOW = 60  # ~2 mesi
    area = ts.rolling(WINDOW, center=True).sum().dropna()  # area sotto la curva nei 60 gg centrati su ogni giorno

    # picchi dell'area e loro estensione (larghezza a metà prominenza)
    # width: alla base (90% della prominenza) un afflusso di D giorni è largo ~D + WINDOW -> scarta afflussi < MIN_DAYS (es. Natale)
    MIN_DAYS = 30
    peaks, _ = find_peaks(area, prominence=0.1 * area.max(), width=WINDOW + MIN_DAYS, rel_height=0.9)
    _, _, left, right = peak_widths(area, peaks, rel_height=0.5)
    periodi = pd.DataFrame({
        "picco": area.index[peaks],
        "inizio": area.index[left.round().astype(int)],
        "fine": area.index[right.round().astype(int)],
    })
    periodi["giorni"] = (periodi["fine"] - periodi["inizio"]).dt.days

    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
        subplot_titles=("Presenze turistiche giornaliere (media mobile 7 gg)", f"Area sotto la curva, finestra mobile {WINDOW} gg"),
    )
    fig.add_trace(go.Scatter(x=ts.index, y=ts, mode="lines", line=dict(width=1.5, color="#1f77b4"), name="Presenze"), row=1, col=1)
    fig.add_trace(go.Scatter(x=area.index, y=area, mode="lines", fill="tozeroy", line=dict(width=1.5, color="#1f77b4"), name=f"Area {WINDOW} gg"), row=2, col=1)
    fig.add_trace(go.Scatter(x=periodi["picco"], y=area.loc[periodi["picco"]], mode="markers", marker=dict(size=8, color="#d62728"), name="Picco"), row=2, col=1)
    for p in periodi.itertuples():
        fig.add_vrect(x0=p.inizio, x1=p.fine, fillcolor="orange", opacity=0.2, line_width=0, layer="below", row="all", col=1)
    fig.update_layout(title=title, template="plotly_white", height=700, showlegend=False, hovermode="x unified")
    fig.show()

show_periodi(ts)

In [ ]:
# normalizzazione min-max per comune: 0 = minimo del comune, 1 = massimo del comune
tourists_attendences_norm = tourists_attendences.copy()
tourists_attendences_norm["value"] = tourists_attendences.groupby("comune")["value"].transform(lambda s: (s - s.min()) / (s.max() - s.min()))
tourists_attendences_norm[tourists_attendences.comune=='ALA'].set_index('date').plot()

In [ ]:
clusters = pd.read_parquet('./DF_CLUSTERS.parquet')

for i in range(1,4):
    cluster_x = clusters[clusters.cluster==i].comune.values
    ts = tourists_attendences_norm[tourists_attendences_norm.comune.isin(cluster_x)].groupby('date')['value'].median()

    show_periodi(ts, title=f"Cluster {i} (mediana dei comuni normalizzati)")

In [ ]:
cluster_x = clusters[clusters.cluster==2].comune.values
for c in cluster_x:
    tourists_attendences_norm[tourists_attendences_norm.comune == c].set_index('date')['value'].plot()

In [ ]:
from statsmodels.tsa.seasonal import STL, seasonal_decompose
import numpy as np

MIN_YOY_CORR = 0.5  # sotto questa soglia l'andamento annuale non si ripete abbastanza: stagionalità non affidabile

seasonal_dfs = []
for comune, g in tourists_attendences.groupby("comune"):
    s = g.set_index("date")["value"].dropna()  # tolti i NaN ai bordi della media mobile
    res = seasonal_decompose(s, period=365, extrapolate_trend="period")

    # quanto si ripete l'andamento annuale: correlazione media tra gli anni del profilo settimanale senza trend
    detrended = s - res.trend
    weekly = detrended.groupby([detrended.index.isocalendar().week.clip(upper=52), detrended.index.year]).mean().unstack().dropna()  # 52 settimane x anni
    yoy_corr = weekly.corr().values[np.triu_indices(weekly.shape[1], 1)].mean()

    seasonal_dfs.append(pd.DataFrame({
        "date": res.seasonal.index, "value": res.seasonal.values, "comune": comune,
        "yoy_corr": yoy_corr, "has_seasonality": yoy_corr >= MIN_YOY_CORR,
    }))

tourists_seasonal = pd.concat(seasonal_dfs, ignore_index=True)
tourists_seasonal

In [ ]:
tourists_seasonal.to_parquet('./tourists_seasonal.parquet')

In [ ]:
px.line(tourists_seasonal[tourists_seasonal.comune=='ALA'].sort_values(by='date'),x='date',y='value')

In [ ]:
i=0
comuni = tourists_attendences.comune.unique()

In [ ]:
i+=1
c = comuni[i]
print(c)
fig2 = px.line(tourists_attendences[tourists_attendences.comune == c], y='value', x='date')
fig1 = px.line(tourists_seasonal[tourists_seasonal.comune == c], y='value', x='date')

fig2.show()
fig1.show()

In [ ]:
i+=1
c = comuni[i]
# c = 'ALA'
print(c)

fig = px.line(tourists_attendences[tourists_attendences.comune == c], y='value', x='date')
fig.add_traces(px.line(tourists_seasonal[tourists_seasonal.comune == c], y='value', x='date').update_traces(line_color='red').data)
fig.update_layout(title=c + ' ' + str(tourists_seasonal[tourists_seasonal.comune==c].has_seasonality.all()))
fig.show()